# Stable Diffusion WebUI reForge on Google Colab

**使い方:**
1. ランタイム → ランタイムのタイプを変更 → GPU（T4推奨）を選択
2. セルを上から順に実行
3. セル5で表示されるGradio URLにアクセス

**保存先:**
- ControlNetモデル・LoRAはGoogle Driveに保存（永続化）
- Checkpoint・VAEはColabローカルに毎回ダウンロード

In [ ]:
#@title セル1: Google Drive接続
from google.colab import drive
from IPython.display import clear_output, display
import ipywidgets as widgets
import os

def inf(msg, style, wdth):
    btn = widgets.Button(description=msg, disabled=True, button_style=style,
                         layout=widgets.Layout(min_width=wdth))
    display(btn)

Shared_Drive = "" #@param {type:"string"}
#@markdown - 共有ドライブを使用しない場合は空白のままにしてください

print("\033[0;33mConnecting...")
drive.mount('/content/gdrive')

if Shared_Drive != "" and os.path.exists("/content/gdrive/Shareddrives"):
    mainpth = "Shareddrives/" + Shared_Drive
else:
    mainpth = "MyDrive"

# DriveにはControlNetとLoRAのみ保存
DRIVE_BASE = f"/content/gdrive/{mainpth}/sd-reforge"
os.makedirs(f"{DRIVE_BASE}/controlnet", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/loras", exist_ok=True)

clear_output()
inf('✔ Done', 'success', '50px')
print(f"Drive path: {DRIVE_BASE}")

In [ ]:
#@title セル2: reForgeインストール & 環境セットアップ
import os, re, subprocess, sys, glob, shutil
from IPython.display import clear_output, display
import ipywidgets as widgets

def inf(msg, style, wdth):
    btn = widgets.Button(description=msg, disabled=True, button_style=style,
                         layout=widgets.Layout(min_width=wdth))
    display(btn)

REFORGE_DIR = "/content/stable-diffusion-webui-reForge"
DRIVE_BASE  = f"/content/gdrive/{mainpth}/sd-reforge"

# ── reForge クローン ──────────────────────────────────────────────
if not os.path.exists(REFORGE_DIR):
    print("reForge をクローン中...")
    subprocess.run([
        "git", "clone", "--depth=1",
        "https://github.com/Panchovix/stable-diffusion-webui-reForge.git",
        REFORGE_DIR
    ], check=True)
    print("クローン完了")
else:
    print("reForge は既にインストール済み")

# ── LoRA networks.py NoneType バグ修正パッチ ─────────────────────
# 問題: LoRA名がディスク上に存在しないとき network_on_disk が None になり
#       load_network() とエラーハンドラの両方でクラッシュする
#   Patch 1: load_network 呼び出し前に None ガードを挿入（スキップ＋警告）
#   Patch 2: except 節の network_on_disk.filename を None 安全な式に置換
_networks_py = f"{REFORGE_DIR}/extensions-builtin/Lora/networks.py"
if os.path.exists(_networks_py):
    with open(_networks_py) as _f:
        _src = _f.read()
    _patched = _src

    def _add_none_guard(m):
        ind = m.group(1)
        return (
            f"{ind}if network_on_disk is None:\n"
            f"{ind}    print(f'[Lora] WARNING: \"{{name}}\" not found on disk, skipping')\n"
            f"{ind}    continue\n"
            f"{ind}net = load_network(name, network_on_disk)"
        )
    if "network_on_disk is None" not in _patched:
        _patched = re.sub(
            r"([ \t]+)(net = load_network\(name, network_on_disk\))",
            _add_none_guard, _patched, count=1
        )
    _patched = _patched.replace(
        'errors.display(e, f"loading network {network_on_disk.filename}")',
        'errors.display(e, f"loading network {network_on_disk.filename if network_on_disk else name}")'
    )
    if _patched != _src:
        with open(_networks_py, "w") as _f:
            _f.write(_patched)
        print("networks.py: NoneType パッチ適用済み")
    else:
        print("networks.py: パッチ不要（既に適用済み）")

# ── ローカルフォルダ作成 ──────────────────────────────────────────
for d in ["models/Stable-diffusion", "models/VAE", "models/Lora",
          "models/ControlNet", "models/ESRGAN", "embeddings", "outputs"]:
    os.makedirs(f"{REFORGE_DIR}/{d}", exist_ok=True)

# ── ControlNet モデルフォルダ → Drive にシンボリックリンク ─────────
controlnet_dir = f"{REFORGE_DIR}/models/ControlNet"
if not os.path.islink(controlnet_dir):
    if os.path.exists(controlnet_dir):
        subprocess.run(["rm", "-rf", controlnet_dir])
    os.symlink(f"{DRIVE_BASE}/controlnet", controlnet_dir)
    print(f"ControlNet リンク作成: {controlnet_dir}")
else:
    print("ControlNet リンク済み")

# sd-webui-controlnet 拡張フォルダもリンク（拡張が存在する場合）
controlnet_ext = f"{REFORGE_DIR}/extensions/sd-webui-controlnet/models"
if os.path.exists(os.path.dirname(controlnet_ext)) and not os.path.islink(controlnet_ext):
    if os.path.exists(controlnet_ext):
        subprocess.run(["rm", "-rf", controlnet_ext])
    os.symlink(f"{DRIVE_BASE}/controlnet", controlnet_ext)
    print(f"ControlNet 拡張リンク作成")

# ── Drive LoRA → ローカル Lora フォルダにシンボリックリンク ────────
drive_lora_dir = f"{DRIVE_BASE}/loras"
local_lora_dir = f"{REFORGE_DIR}/models/Lora"
drive_loras = [f for f in os.listdir(drive_lora_dir)
               if f.endswith('.safetensors') or f.endswith('.pt')]
for fname in drive_loras:
    dst = f"{local_lora_dir}/{fname}"
    if not os.path.exists(dst):
        os.symlink(f"{drive_lora_dir}/{fname}", dst)
        print(f"Drive LoRA リンク: {fname}")
print(f"Drive からリンクした LoRA: {len(drive_loras)} 個")

# ── wandb を完全削除（reForge 起動時の競合を防ぐ）─────────────────
print("wandb を削除中...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "wandb"],
               check=False, capture_output=True)
for _d in glob.glob("/usr/local/lib/python*/dist-packages/wandb"):
    shutil.rmtree(_d, ignore_errors=True)
print("wandb 削除完了")

# ── 必要な依存ライブラリの事前インストール ───────────────────────
# accelerate>=0.35.0: peft が clear_device_cache を要求するため 0.33.0 では不足
print("\n依存関係インストール中（数分かかります）...")
deps = [
    "accelerate>=0.35.0",
    "diffusers==0.30.3",
    "invisible-watermark",
    "xformers",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps, check=False)

clear_output()
inf('✔ Done', 'success', '50px')
print(f"REFORGE_DIR: {REFORGE_DIR}")
print(f"Drive LoRA: {len(drive_loras)} 個")

In [ ]:
#@title セル3: モデル・VAEダウンロード

#@markdown ## Civitai API Key
#@markdown Civitai のモデルを DL する場合は API キーを入力してください
#@markdown（取得: https://civitai.com/user/account）
civitai_api_key = "" #@param {type:"string"}

import os, subprocess

REFORGE_DIR = "/content/stable-diffusion-webui-reForge"
MODEL_DIR   = f"{REFORGE_DIR}/models/Stable-diffusion"
VAE_DIR     = f"{REFORGE_DIR}/models/VAE"

def _wget(url, dest_dir, filename=""):
    """指定ファイルが既存なら skip、なければ wget でダウンロード"""
    if filename:
        fpath = f"{dest_dir}/{filename}"
        if os.path.exists(fpath) and os.path.getsize(fpath) > 1024 * 1024:
            print(f"  ✔ スキップ（既存）: {filename}")
            return
    label = filename or url.split("/")[-1]
    print(f"  DL中: {label}")
    r = subprocess.run(
        ["wget", "-q", "--content-disposition", f"--directory-prefix={dest_dir}", url],
        capture_output=True, text=True
    )
    print("  ✔ 完了" if r.returncode == 0 else f"  ✗ 失敗: {r.stderr[:200]}")

def civitai_url(model_id):
    url = f"https://civitai.com/api/download/models/{model_id}"
    return url + (f"?token={civitai_api_key}" if civitai_api_key else "")

# ──────────────────────────────────────────────────────────────────
#@markdown ---
#@markdown ## Checkpoint（SD 1.5 系）

#@markdown ### chilled_remix v2（実写系）
use_chilled_remix_v2 = False #@param {type:"boolean"}
if use_chilled_remix_v2:
    _wget(
        "https://huggingface.co/sazyou-roukaku/chilled_remix/resolve/main/chilled_remix_v2.safetensors",
        MODEL_DIR, "chilled_remix_v2.safetensors"
    )

#@markdown ### Realistic Vision V5.1（実写系 / Civitai）
use_realistic_vision = False #@param {type:"boolean"}
if use_realistic_vision:
    _wget(civitai_url(130072), MODEL_DIR, "realisticVisionV51.safetensors")

#@markdown ### DreamShaper v8（汎用）
use_dreamshaper = False #@param {type:"boolean"}
if use_dreamshaper:
    _wget(civitai_url(128713), MODEL_DIR, "dreamshaper_8.safetensors")

#@markdown ---
#@markdown ## VAE

#@markdown ### vae-ft-mse-840000（SD 1.5 標準 VAE）
use_vae_sd15 = True #@param {type:"boolean"}
if use_vae_sd15:
    _wget(
        "https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors",
        VAE_DIR, "vae-ft-mse-840000-ema-pruned.safetensors"
    )

# ── フォルダ状況表示 ─────────────────────────────────────────────
print("\n--- モデルフォルダ ---")
for label, path in [("Checkpoint", MODEL_DIR), ("VAE", VAE_DIR)]:
    files = [f for f in os.listdir(path) if os.path.isfile(f"{path}/{f}")]
    print(f"  {label}: {len(files)} 個")
    for f in files:
        mb = os.path.getsize(f"{path}/{f}") / 1024 / 1024
        print(f"    - {f} ({mb:.1f} MB)")

In [ ]:
#@title セル4: LoRAダウンロード（Civitai → ローカル or Drive）
import os, subprocess

REFORGE_DIR = "/content/stable-diffusion-webui-reForge"
LORA_DIR    = f"{REFORGE_DIR}/models/Lora"
os.makedirs(LORA_DIR, exist_ok=True)

# civitai_api_key はセル3で定義済み

def download_lora(model_id, hint_name="", to_drive=False):
    """
    hint_name を含むファイルが LORA_DIR に存在すればスキップ。
    to_drive=True のとき Drive にも保存しリンクを張る。
    """
    if hint_name:
        existing = [f for f in os.listdir(LORA_DIR) if hint_name.lower() in f.lower()]
        if existing:
            print(f"  ✔ スキップ（既存）: {existing[0]}")
            return
    url = f"https://civitai.com/api/download/models/{model_id}"
    if civitai_api_key:
        url += f"?token={civitai_api_key}"
    dest = f"{DRIVE_BASE}/loras" if to_drive else LORA_DIR
    print(f"  DL中: {hint_name or f'model_id={model_id}'}")
    r = subprocess.run(
        ["wget", "-q", "--content-disposition", f"--directory-prefix={dest}", url],
        capture_output=True, text=True
    )
    print("  ✔ 完了" if r.returncode == 0 else f"  ✗ 失敗: {r.stderr[:200]}")
    # Drive 保存の場合、ローカルへシンボリックリンク
    if to_drive and r.returncode == 0:
        for f in os.listdir(dest):
            if hint_name.lower() in f.lower():
                dst = f"{LORA_DIR}/{f}"
                if not os.path.exists(dst):
                    os.symlink(f"{dest}/{f}", dst)
                break

#@markdown ---
#@markdown ## LoRA選択

#@markdown ### Aesthetic Quality Modifiers - Masterpiece
#@markdown [公式ページ](https://civitai.com/models/929497?modelVersionId=1050644)
use_Masterpiece = True #@param {type:"boolean"}
if use_Masterpiece:
    download_lora(1050644, hint_name="masterpiece", to_drive=True)

#@markdown ### Detail Tweaker LoRA
use_detail_tweaker = False #@param {type:"boolean"}
if use_detail_tweaker:
    download_lora(58390, hint_name="detail_tweaker")

# ── 認識されるLoRA一覧 ───────────────────────────────────────────
print("\n--- 認識される LoRA 一覧 ---")
loras = sorted([f for f in os.listdir(LORA_DIR)
                if f.endswith('.safetensors') or f.endswith('.pt')])
for f in loras:
    tag  = "[Drive]" if os.path.islink(f"{LORA_DIR}/{f}") else "[Local]"
    size = os.path.getsize(f"{LORA_DIR}/{f}") / 1024 / 1024
    print(f"  {tag} {f} ({size:.1f} MB)")
print(f"合計: {len(loras)} 個")

In [ ]:
#@title セル5: SD WebUI reForge 起動（Gradio）

#@markdown ## 起動オプション
use_xformers   = True  #@param {type:"boolean"}
#@markdown - メモリ効率が上がる（GPU が xformers 対応の場合）

use_half_vae   = True  #@param {type:"boolean"}
#@markdown - False にすると VRAM 消費増だが VAE 品質安定

tunnel_type    = "gradio" #@param ["gradio", "cloudflared"]
#@markdown - `gradio`: Gradio 組み込み共有 URL（*.gradio.live）
#@markdown - `cloudflared`: Cloudflare Tunnel URL（より安定）

import os, subprocess, threading, time, sys, re
from IPython.display import display, HTML
import ipywidgets as widgets

REFORGE_DIR = "/content/stable-diffusion-webui-reForge"
LOG_FILE    = "/content/reforge.log"
PORT        = 7860

# ── 起動コマンド構築 ─────────────────────────────────────────────
cmd = [
    sys.executable, f"{REFORGE_DIR}/launch.py",
    "--skip-torch-cuda-test",
    "--disable-safe-unpickle",
    "--enable-insecure-extension-access",
    "--api",
    "--port", str(PORT),
    "--lora-dir", f"{REFORGE_DIR}/models/Lora",
]

if not use_half_vae:
    cmd.append("--no-half-vae")
if use_xformers:
    cmd.append("--xformers")
if tunnel_type == "gradio":
    cmd.append("--share")          # Gradio 組み込み共有
else:
    cmd.append("--listen")         # cloudflared 用にすべての IF で待ち受け

# ── cloudflared インストール（選択時のみ）────────────────────────
if tunnel_type == "cloudflared":
    if not os.path.exists("/usr/local/bin/cloudflared"):
        print("cloudflared をインストール中...")
        subprocess.run(
            "curl -sL https://github.com/cloudflare/cloudflared/releases/latest"
            "/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared"
            " && chmod +x /usr/local/bin/cloudflared",
            shell=True, check=True
        )
        print("cloudflared インストール完了")

# ── WebUI プロセス起動 ───────────────────────────────────────────
def run_webui():
    with open(LOG_FILE, "w") as log:
        subprocess.run(cmd, stdout=log, stderr=log, cwd=REFORGE_DIR)

webui_thread = threading.Thread(target=run_webui, daemon=True)
webui_thread.start()

# ── cloudflared トンネル（選択時のみ）───────────────────────────
tunnel_url = None

def run_cloudflare_tunnel():
    global tunnel_url
    # WebUI が起動するまで少し待つ
    time.sleep(20)
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        m = re.search(r'https://[^\s]+\.trycloudflare\.com', line)
        if m:
            tunnel_url = m.group(0)
            print(f"\n\033[92m✔ cloudflared URL: {tunnel_url}\033[0m\n")
            break

if tunnel_type == "cloudflared":
    threading.Thread(target=run_cloudflare_tunnel, daemon=True).start()

# ── 起動待機・URL 検出 ───────────────────────────────────────────
print("WebUI 起動中... （初回は5〜10分かかる場合があります）")
print(f"起動コマンド: {' '.join(cmd[:6])} ...\n")

gradio_url  = None
start_time  = time.time()
TIMEOUT_SEC = 600  # 最大10分待機

while time.time() - start_time < TIMEOUT_SEC:
    time.sleep(3)
    if not os.path.exists(LOG_FILE):
        continue
    with open(LOG_FILE) as f:
        log = f.read()

    # Gradio 共有 URL を検出
    if tunnel_type == "gradio":
        m = re.search(r'Running on public URL: (https://[^\s]+\.gradio\.live)', log)
        if m:
            gradio_url = m.group(1)
            break

    # ローカル起動確認（cloudflared モード）
    if tunnel_type == "cloudflared":
        if "Running on local URL" in log:
            break

    # 共通エラー検出
    if "Traceback (most recent call last)" in log and "Error" in log:
        # 致命的エラーの可能性があるが起動継続中の場合もあるため警告のみ
        elapsed = int(time.time() - start_time)
        if elapsed % 30 == 0:
            print(f"  [{elapsed}s] ログにエラー検出（継続待機中）...")

    elapsed = int(time.time() - start_time)
    if elapsed % 60 == 0 and elapsed > 0:
        print(f"  [{elapsed}s] 待機中...")

# ── 結果表示 ─────────────────────────────────────────────────────
print("\n" + "="*60)
if gradio_url:
    print(f"\033[92m✔ Gradio 起動完了！\033[0m")
    display(HTML(f'<h3>✅ <a href="{gradio_url}" target="_blank">{gradio_url}</a></h3>'))
elif tunnel_url:
    print(f"\033[92m✔ WebUI 起動完了！\033[0m")
    display(HTML(f'<h3>✅ <a href="{tunnel_url}" target="_blank">{tunnel_url}</a></h3>'))
else:
    elapsed = int(time.time() - start_time)
    print(f"⚠ {elapsed}秒経過 — URL を検出できませんでした。")
    print("  以下のログを確認してください:")

print("\n--- 最新ログ（末尾 2000 文字）---")
with open(LOG_FILE) as f:
    print(f.read()[-2000:])

In [ ]:
#@title セル6（任意）: ログのリアルタイム確認
import time, os

LOG_FILE = "/content/reforge.log"
lines_shown = 0

for _ in range(200):          # 最大 200 × 3s = 10 分
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE) as f:
            lines = f.readlines()
        new_lines = lines[lines_shown:]
        if new_lines:
            print("".join(new_lines), end="")
            lines_shown = len(lines)
    time.sleep(3)